# AI Analytics Pipeline for Poultry Farm IoT

## Smart Environmental Monitoring with Anomaly Detection, Forecasting, and Risk Scoring

This notebook implements the AI analytics layer for the poultry farm monitoring system, including:
1. **Anomaly Detection** - Z-score baseline, Isolation Forest, LOF, and ensemble
2. **Forecasting** - Multi-sensor early warning (15/30/60 min horizons)
3. **Risk Scoring** - Interpretable shed health index
4. **Environmental State Clustering** - Pattern discovery for farm conditions

---

**Academic Positioning**: This AI layer extends the threshold-based monitoring system with data-driven anomaly detection and short-horizon forecasting to enable proactive intervention.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import json
from datetime import datetime

# Add parent directory to path
sys.path.append(os.path.dirname(os.getcwd()))

# Import custom modules
from anomaly_detection import (
    AnomalyDetector, AnomalyInjector, FeatureEngineer, run_anomaly_detection_pipeline
)
from forecasting import (
    MultiSensorForecaster, create_forecast_horizon_comparison, run_forecasting_pipeline
)
from risk_scoring import (
    RiskScorer, EnvironmentalStateClusterer, run_risk_scoring_pipeline
)

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Libraries loaded successfully!")

## 1. Data Loading and Exploration

In [ ]:
# Load environmental sensor data
DATA_PATH = '../DATA/environment_dataset.csv'
OUTPUT_DIR = 'results'
PLOTS_DIR = 'plots'
TABLES_DIR = 'tables'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'])

sensor_cols = ['temperature', 'humidity', 'co2', 'ammonia']

print(f"Dataset shape: {df.shape}")
print(f"Time range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"\nSensor Statistics:")
df[sensor_cols].describe()

In [ ]:
# Plot raw sensor data
fig, axes = plt.subplots(4, 1, figsize=(14, 16))

for idx, sensor in enumerate(sensor_cols):
    axes[idx].plot(df['timestamp'], df[sensor], linewidth=0.8, color='steelblue')
    axes[idx].set_ylabel(sensor.title())
    axes[idx].set_title(f'{sensor.title()} Over Time')
    axes[idx].grid(True, alpha=0.3)

axes[-1].set_xlabel('Timestamp')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'raw_sensor_data.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2. Anomaly Detection

### Method Overview
- **Z-Score Baseline**: Rolling statistical detection (|z| > 3)
- **Isolation Forest**: Unsupervised tree-based anomaly detection
- **Local Outlier Factor**: Density-based anomaly detection
- **Ensemble**: Weighted combination of all methods

### Synthetic Anomaly Injection
Since labeled anomalies are unavailable, we inject synthetic anomalies:
- Spikes: Sudden large deviations
- Drift: Gradual sensor drift
- Stuck: Sensor stuck at fixed value
- Correlated: Cross-sensor anomalies (CO2 + ammonia)

In [ ]:
# Run anomaly detection pipeline
print("="*60)
print("ANOMALY DETECTION PIPELINE")
print("="*60)

# Inject synthetic anomalies for evaluation
injector = AnomalyInjector(contamination=0.05, seed=42)
df_anomalous, anomaly_labels, anomaly_info = injector.inject_anomalies(
    df, sensor_cols, anomaly_types=['spike', 'drift', 'stuck', 'correlated']
)

print(f"Total samples: {len(df_anomalous)}")
print(f"Injected anomalies: {anomaly_labels.sum()} ({100*anomaly_labels.mean():.2f}%)")

# Save anomaly info
with open(os.path.join(OUTPUT_DIR, 'injected_anomalies.json'), 'w') as f:
    json.dump(anomaly_info, f, indent=2)

In [ ]:
# Fit anomaly detector on clean data
safe_ranges = {
    'temperature': (18, 32),
    'humidity': (40, 75),
    'co2': (400, 1500),
    'ammonia': (0, 25)
}

detector = AnomalyDetector(contamination=0.05)
detector.fit(df, sensor_cols, safe_ranges)

# Predict on anomalous data
z_labels, z_scores = detector.predict_zscore(df_anomalous)
if_labels, if_scores = detector.predict_isolation_forest(df_anomalous)
lof_labels, lof_scores = detector.predict_lof(df_anomalous)
ensemble_labels, ensemble_scores = detector.predict_ensemble(df_anomalous)

In [ ]:
# Evaluate all methods
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

def evaluate_method(y_true, y_pred, name):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    far = fp / (fp + tn) if (fp + tn) > 0 else 0  # False alarm rate
    
    return {
        'Method': name,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'False Alarm Rate': far,
        'TP': tp, 'FP': fp, 'TN': tn, 'FN': fn
    }

metrics = [
    evaluate_method(anomaly_labels, z_labels, 'Z-Score'),
    evaluate_method(anomaly_labels, if_labels, 'Isolation Forest'),
    evaluate_method(anomaly_labels, lof_labels, 'LOF'),
    evaluate_method(anomaly_labels, ensemble_labels, 'Ensemble')
]

metrics_df = pd.DataFrame(metrics)
print("\n" + "="*60)
print("ANOMALY DETECTION PERFORMANCE COMPARISON")
print("="*60)
print(metrics_df.to_string(index=False))

In [ ]:
# Save metrics table
metrics_df.to_csv(os.path.join(TABLES_DIR, 'anomaly_detection_comparison.csv'), index=False)

# Create publication-ready table
latex_table = r"""\begin{table}[h]
\centering
\caption{Anomaly Detection Performance Comparison}
\label{tab:anomaly_comparison}
\begin{tabular}{lcccc}
\hline
\textbf{Method} & \textbf{Precision} & \textbf{Recall} & \textbf{F1-Score} & \textbf{False Alarm Rate} \\
\hline
"""

for _, row in metrics_df.iterrows():
    latex_table += f"{row['Method']} & {row['Precision']:.4f} & {row['Recall']:.4f} & {row['F1-Score']:.4f} & {row['False Alarm Rate']:.4f} \\\\\n"

latex_table += r"""\hline
\end{tabular}
\end{table}"""

with open(os.path.join(TABLES_DIR, 'anomaly_detection_comparison.tex'), 'w') as f:
    f.write(latex_table)

print("\nLaTeX table saved to:", os.path.join(TABLES_DIR, 'anomaly_detection_comparison.tex'))

In [ ]:
# Visualize anomaly detection results
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot 1: True anomalies vs detected
ax1 = axes[0, 0]
ax1.fill_between(range(len(anomaly_labels)), 0, anomaly_labels, alpha=0.5, label='True Anomalies', color='red')
ax1.fill_between(range(len(ensemble_labels)), 0, ensemble_labels, alpha=0.5, label='Ensemble Detected', color='blue')
ax1.set_xlabel('Sample Index')
ax1.set_ylabel('Anomaly Label')
ax1.set_title('True vs Detected Anomalies (Ensemble)')
ax1.legend()
ax1.set_ylim(-0.1, 1.1)

# Plot 2: Ensemble scores with threshold
ax2 = axes[0, 1]
ax2.plot(ensemble_scores, linewidth=0.8, color='steelblue', label='Ensemble Score')
ax2.axhline(y=0.5, color='red', linestyle='--', label='Threshold (0.5)')
ax2.fill_between(range(len(ensemble_scores)), ensemble_scores, 0.5, 
                  where=(ensemble_scores > 0.5), alpha=0.3, color='red')
ax2.set_xlabel('Sample Index')
ax2.set_ylabel('Anomaly Score')
ax2.set_title('Ensemble Anomaly Scores')
ax2.legend()

# Plot 3: F1-score comparison
ax3 = axes[1, 0]
methods = metrics_df['Method'].values
f1_scores = metrics_df['F1-Score'].values
colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']
bars = ax3.bar(methods, f1_scores, color=colors)
ax3.set_ylabel('F1-Score')
ax3.set_title('F1-Score Comparison Across Methods')
ax3.set_ylim(0, 1)
for bar, val in zip(bars, f1_scores):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             f'{val:.3f}', ha='center', va='bottom', fontsize=10)

# Plot 4: Anomaly type breakdown
ax4 = axes[1, 1]
anomaly_types = list(anomaly_info.keys())
counts = [len(anomaly_info[t]) for t in anomaly_types]
ax4.bar(anomaly_types, counts, color=['#e74c3c', '#3498db', '#2ecc71', '#f39c12'])
ax4.set_ylabel('Number of Events')
ax4.set_title('Injected Anomaly Types Distribution')
for bar, count in zip(ax4.patches, counts):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             str(count), ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'anomaly_detection_results.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3. Multi-Sensor Forecasting

### Method: Random Forest with Lag Features
- Lag features: 1, 2, 3, 5, 10, 15 steps
- Rolling statistics: mean, std
- Time features: hour encoding
- Exogenous variables: other sensors

### Forecast Horizons: 15, 30, 60 minutes

In [ ]:
# Compare forecast horizons
print("="*60)
print("FORECASTING HORIZON COMPARISON")
print("="*60)

horizons = [15, 30, 60]
horizon_results = create_forecast_horizon_comparison(df, sensor_cols, horizons)

# Save results
horizon_results.to_csv(os.path.join(OUTPUT_DIR, 'horizon_comparison.csv'), index=False)

print(horizon_results.to_string(index=False))

In [ ]:
# Fit final forecaster with 15-min horizon (best for early warning)
forecaster = MultiSensorForecaster(forecast_horizon=15, model_type='rf')
forecaster.fit(df, sensor_cols)

# Generate predictions
predictions = forecaster.predict(df)

# Evaluate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

forecast_metrics = []
for sensor in sensor_cols:
    # Align predictions (shorter due to lag features)
    n_pred = len(predictions[sensor])
    y_true = df[sensor].iloc[-n_pred:].values
    y_pred = predictions[sensor]
    
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100
    
    forecast_metrics.append({
        'Sensor': sensor.title(),
        'MAE': mae,
        'RMSE': rmse,
        'R²': r2,
        'MAPE (%)': mape
    })

forecast_metrics_df = pd.DataFrame(forecast_metrics)
print("\n" + "="*60)
print("FORECASTING PERFORMANCE (15-min horizon)")
print("="*60)
print(forecast_metrics_df.to_string(index=False))

In [ ]:
# Save forecasting metrics table
forecast_metrics_df.to_csv(os.path.join(TABLES_DIR, 'forecasting_performance.csv'), index=False)

# Create LaTeX table
latex_forecast = r"""\begin{table}[h]
\centering
\caption{Multi-Sensor Forecasting Performance (15-minute horizon)}
\label{tab:forecasting}
\begin{tabular}{lcccc}
\hline
\textbf{Sensor} & \textbf{MAE} & \textbf{RMSE} & \textbf{R²} & \textbf{MAPE (\%)} \\
\hline
"""

for _, row in forecast_metrics_df.iterrows():
    latex_forecast += f"{row['Sensor']} & {row['MAE']:.4f} & {row['RMSE']:.4f} & {row['R²']:.4f} & {row['MAPE (%)']:.2f} \\\\\n"

latex_forecast += r"""\hline
\end{tabular}
\end{table}"""

with open(os.path.join(TABLES_DIR, 'forecasting_performance.tex'), 'w') as f:
    f.write(latex_forecast)

print("LaTeX table saved to:", os.path.join(TABLES_DIR, 'forecasting_performance.tex'))

In [ ]:
# Visualize forecasting results
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for idx, sensor in enumerate(sensor_cols):
    ax = axes[idx // 2, idx % 2]
    
    # Plot last 200 points for clarity
    n_plot = 200
    y_true = df[sensor].iloc[-n_pred:].values[-n_plot:]
    y_pred = predictions[sensor][-n_plot:]
    x = range(n_plot)
    
    ax.plot(x, y_true, label='Actual', linewidth=1.5, color='steelblue')
    ax.plot(x, y_pred, label='Predicted (15-min ahead)', linewidth=1.5, color='coral', linestyle='--')
    ax.set_xlabel('Time Step')
    ax.set_ylabel(sensor.title())
    ax.set_title(f'{sensor.title()} - Actual vs Predicted')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'forecasting_results.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Risk Scoring System

### Shed Health Risk Index
Composite risk score (0-1) based on:
- Deviation from optimal ranges
- Persistence of abnormal conditions
- Weighted sensor contributions

### Risk Levels:
- **Normal**: risk < 0.25
- **Warning**: 0.25 ≤ risk < 0.5
- **Critical**: risk ≥ 0.5

In [ ]:
# Compute risk scores
print("="*60)
print("RISK SCORING SYSTEM")
print("="*60)

scorer = RiskScorer()
df_risk = scorer.compute_risk_score(df, sensor_cols)

# Risk distribution
print("\nRisk Level Distribution:")
risk_dist = df_risk['risk_level'].value_counts()
print(risk_dist)

# Save risk scores
df_risk.to_csv(os.path.join(OUTPUT_DIR, 'risk_scores.csv'), index=False)

In [ ]:
# Generate recommendations
recommendations = scorer.generate_recommendations(df_risk, sensor_cols)
recommendations.to_csv(os.path.join(OUTPUT_DIR, 'recommendations.csv'), index=False)

# Show sample recommendations
print("\nSample Recommendations (Critical/Warning periods):")
critical_recs = recommendations[recommendations['risk_level'].isin(['Warning', 'Critical'])].head(10)
print(critical_recs[['timestamp', 'risk_level', 'recommendations']].to_string(index=False))

In [ ]:
# Visualize risk scores
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Plot 1: Composite risk over time
ax1 = axes[0]
ax1.fill_between(range(len(df_risk)), df_risk['composite_risk'], 0, 
                  color='steelblue', alpha=0.6)
ax1.axhline(y=0.25, color='green', linestyle='--', label='Normal/Warning threshold')
ax1.axhline(y=0.5, color='orange', linestyle='--', label='Warning/Critical threshold')
ax1.axhline(y=0.75, color='red', linestyle='--', label='High risk')
ax1.set_ylabel('Composite Risk Score')
ax1.set_title('Shed Health Risk Index Over Time')
ax1.legend(loc='upper right')
ax1.set_ylim(0, 1)

# Plot 2: Risk level timeline
ax2 = axes[1]
risk_colors = {'Normal': 'green', 'Warning': 'orange', 'Critical': 'red'}
risk_numeric = df_risk['risk_level'].map({'Normal': 0, 'Warning': 1, 'Critical': 2})
scatter = ax2.scatter(range(len(risk_numeric)), risk_numeric, 
                      c=risk_numeric, cmap='RdYlGn_r', s=10, alpha=0.6)
ax2.set_ylabel('Risk Level')
ax2.set_title('Risk Level Timeline')
ax2.set_yticks([0, 1, 2])
ax2.set_yticklabels(['Normal', 'Warning', 'Critical'])
ax2.set_ylim(-0.5, 2.5)

# Plot 3: Individual sensor risks (last 500 points)
ax3 = axes[2]
n_plot = 500
risk_cols = [f'{s}_risk' for s in sensor_cols]
for sensor in sensor_cols:
    ax3.plot(df_risk[f'{sensor}_risk'].values[-n_plot:], label=sensor.title(), linewidth=1)
ax3.set_xlabel('Time Step')
ax3.set_ylabel('Sensor Risk')
ax3.set_title('Individual Sensor Risk Contributions')
ax3.legend(loc='upper right')
ax3.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'risk_scoring_results.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Environmental State Clustering

**Important**: This clusters farm environmental conditions, NOT sensor-reference validation pairs.

### Features:
- Sensor values (temperature, humidity, CO2, ammonia)
- Rolling variance (instability indicator)
- Time of day encoding

### Possible Cluster Interpretations:
- Normal/Comfortable state
- Poor ventilation state
- Heat-stress state
- Gas-accumulation state

In [ ]:
# Environmental state clustering
print("="*60)
print("ENVIRONMENTAL STATE CLUSTERING")
print("="*60)

clusterer = EnvironmentalStateClusterer(n_clusters=4)
cluster_labels, semantic_labels = clusterer.fit(df, sensor_cols)

# Add to dataframe
df['cluster'] = cluster_labels
df['cluster_label'] = semantic_labels

# Save results
df.to_csv(os.path.join(OUTPUT_DIR, 'data_with_states.csv'), index=False)

# Cluster profiles
cluster_summary = clusterer.get_cluster_summary()
cluster_summary.to_csv(os.path.join(OUTPUT_DIR, 'cluster_profiles.csv'))

print("\nCluster Profiles:")
print(cluster_summary.to_string())

# Cluster distribution
print("\nCluster Distribution:")
cluster_dist = pd.Series(semantic_labels).value_counts()
print(cluster_dist)

In [ ]:
# Visualize clustering
from sklearn.decomposition import PCA

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Cluster distribution
ax1 = axes[0]
cluster_counts = df['cluster_label'].value_counts()
colors = ['#2ecc71', '#f39c12', '#e74c3c', '#3498db']
bars = ax1.bar(range(len(cluster_counts)), cluster_counts.values, color=colors[:len(cluster_counts)])
ax1.set_xlabel('Environmental State')
ax1.set_ylabel('Frequency')
ax1.set_title('Environmental State Distribution')
ax1.set_xticks(range(len(cluster_counts)))
ax1.set_xticklabels(cluster_counts.index, rotation=45, ha='right')
for bar, count in zip(bars, cluster_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             str(count), ha='center', va='bottom', fontsize=10)

# Plot 2: PCA visualization
ax2 = axes[1]
features_for_pca = clusterer.create_state_features(df, sensor_cols)
pca = PCA(n_components=2)
pca_result = pca.fit_transform(features_for_pca)

scatter = ax2.scatter(pca_result[:, 0], pca_result[:, 1], 
                      c=df['cluster'], cmap='viridis', alpha=0.5, s=10)
ax2.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
ax2.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
ax2.set_title('Environmental States (PCA Projection)')
plt.colorbar(scatter, ax=ax2, label='Cluster ID')

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'clustering_results.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Summary Tables for Report

In [ ]:
# Create comprehensive summary table
summary_data = {
    'AI Module': ['Anomaly Detection', 'Forecasting', 'Risk Scoring', 'State Clustering'],
    'Method': ['Ensemble (Z-score + IF + LOF)', 'Random Forest + Lag Features', 
               'Weighted Multi-Sensor Index', 'K-Means (k=4)'],
    'Best Performance': [
        f"F1 = {metrics_df.loc[metrics_df['Method'] == 'Ensemble', 'F1-Score'].values[0]:.4f}",
        f"R² = {forecast_metrics_df['R²'].mean():.4f}",
        f"Normal: {risk_dist.get('Normal', 0)} ({100*risk_dist.get('Normal', 0)/len(df_risk):.1f}%)",
        f"4 distinct states identified"
    ],
    'Output': ['Anomaly score, label, explanation', '15-min ahead prediction',
               'Risk level (Normal/Warning/Critical)', 'Environmental state label'],
    'Dashboard Use': ['Real-time alerts', 'Early warning', 'Health index display', 'Pattern recognition']
}

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(os.path.join(TABLES_DIR, 'ai_analytics_summary.csv'), index=False)

print("AI Analytics Summary:")
print(summary_df.to_string(index=False))

In [ ]:
# Create LaTeX summary table
latex_summary = r"""\begin{table}[h]
\centering
\caption{AI Analytics Modules Summary}
\label{tab:ai_summary}
\begin{tabular}{l|p{4cm}|p{3cm}|p{3cm}}
\hline
\textbf{Module} & \textbf{Method} & \textbf{Performance} & \textbf{Output} \\
\hline
Anomaly Detection & Ensemble (Z-score + IF + LOF) & """ + f"F1 = {metrics_df.loc[metrics_df['Method'] == 'Ensemble', 'F1-Score'].values[0]:.4f}" + r""" & Anomaly score, label \\
\hline
Forecasting & Random Forest + Lag Features & """ + f"R² = {forecast_metrics_df['R²'].mean():.4f}" + r""" & 15-min ahead prediction \\
\hline
Risk Scoring & Weighted Multi-Sensor Index & """ + f"Normal: {100*risk_dist.get('Normal', 0)/len(df_risk):.1f}\%" + r""" & Normal/Warning/Critical \\
\hline
State Clustering & K-Means (k=4) & 4 states identified & Environmental state \\
\hline
\end{tabular}
\end{table}"""

with open(os.path.join(TABLES_DIR, 'ai_analytics_summary.tex'), 'w') as f:
    f.write(latex_summary)

print("\nLaTeX summary saved to:", os.path.join(TABLES_DIR, 'ai_analytics_summary.tex'))

## 7. Final Results Summary

In [ ]:
print("="*60)
print("AI ANALYTICS PIPELINE COMPLETE")
print("="*60)

print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"Plots directory: {PLOTS_DIR}")
print(f"Tables directory: {TABLES_DIR}")

print("\nGenerated Files:")
for root, dirs, files in os.walk(OUTPUT_DIR):
    for file in files:
        print(f"  - {os.path.join(root, file)}")

for root, dirs, files in os.walk(PLOTS_DIR):
    for file in files:
        print(f"  - {os.path.join(root, file)}")

for root, dirs, files in os.walk(TABLES_DIR):
    for file in files:
        print(f"  - {os.path.join(root, file)}")

print("\n" + "="*60)
print("KEY FINDINGS")
print("="*60)
print(f"""
1. ANOMALY DETECTION:
   - Best method: Ensemble (F1 = {metrics_df.loc[metrics_df['Method'] == 'Ensemble', 'F1-Score'].values[0]:.4f})
   - Detects: spikes, drift, stuck sensors, cross-sensor anomalies
   - Suitable for: real-time alerting

2. FORECASTING:
   - Horizon: 15 minutes (optimal for early warning)
   - Average R²: {forecast_metrics_df['R²'].mean():.4f}
   - Enables: proactive intervention before conditions worsen

3. RISK SCORING:
   - Normal periods: {risk_dist.get('Normal', 0)} ({100*risk_dist.get('Normal', 0)/len(df_risk):.1f}%)
   - Warning periods: {risk_dist.get('Warning', 0)} ({100*risk_dist.get('Warning', 0)/len(df_risk):.1f}%)
   - Critical periods: {risk_dist.get('Critical', 0)} ({100*risk_dist.get('Critical', 0)/len(df_risk):.1f}%)
   - Output: Interpretable health index for dashboard

4. ENVIRONMENTAL STATE CLUSTERING:
   - 4 distinct environmental states identified
   - Useful for: pattern recognition, operational insights
   - Note: Clusters farm conditions, NOT sensor-reference validation
""")

print("="*60)